# Aromatase ML Model Building (Colab GPU-Optimized)

Train 16 regression models across 12 fingerprint types × 2 split strategies = 384 model fits.

**Optimizations:**
- GPU acceleration via cuML (Ridge, Lasso, ElasticNet, KNN, SVR, Random Forest) and XGBoost (`device="cuda"`)
- Parallel CV folds via `n_jobs=-1` for CPU-bound sklearn models
- Memory-efficient: float32 dtypes, sparse matrices for low-density binary FPs
- Incremental checkpointing (resumable)

**Runtime:** Select **G4 GPU** (RTX Pro 6000 Blackwell, 96GB VRAM) for best performance.  
Also works on T4/L4 with automatic fallback.

## 1. Setup & Install

In [ ]:
%%capture
# Install RAPIDS cuML for GPU-accelerated ML
# This works on Colab with T4, L4, A100, or G4 GPU runtimes
!pip install --extra-index-url=https://pypi.nvidia.com cuml-cu12 cudf-cu12 --quiet
!pip install xgboost tqdm --quiet

In [ ]:
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Check GPU
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")

# Check cuML availability
try:
    import cuml
    CUML_AVAILABLE = True
    print(f"cuML {cuml.__version__} available - GPU acceleration enabled")
except ImportError:
    CUML_AVAILABLE = False
    print("cuML not available - falling back to sklearn (CPU only)")

import xgboost
print(f"XGBoost {xgboost.__version__}")

## 2. Clone Repository & Configure Paths

In [ ]:
import os

# Clone the aromatase repository from GitHub
REPO_URL = 'https://github.com/dataprofessor/aromatase.git'
REPO_DIR = '/content/aromatase'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Configure paths
SPLITS_DIR = os.path.join(REPO_DIR, 'data', 'splits')
CURATED_PATH = os.path.join(REPO_DIR, 'data', 'processed', 'aromatase_bioactivity_curated.csv')
OUTPUT_DIR = os.path.join(REPO_DIR, 'data', 'models')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths exist
assert os.path.isdir(SPLITS_DIR), f"Splits dir not found: {SPLITS_DIR}"
assert os.path.isfile(CURATED_PATH), f"Curated file not found: {CURATED_PATH}"

split_files = sorted(os.listdir(SPLITS_DIR))
print(f"Found {len(split_files)} split files in {SPLITS_DIR}")
print(f"Output will be saved to: {OUTPUT_DIR}")


## 3. Data Loading Utilities

In [ ]:
import gc
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

RANDOM_STATE = 42
N_FOLDS = 10


def load_curated_targets(path):
    """Load pchembl_value lookup from curated bioactivity data."""
    df = pd.read_csv(path, usecols=['molecule_chembl_id', 'pchembl_value'])
    df = df.dropna(subset=['pchembl_value'])
    # Keep first occurrence per molecule
    df = df.drop_duplicates(subset='molecule_chembl_id', keep='first')
    return df.set_index('molecule_chembl_id')['pchembl_value'].to_dict()


def load_split_data(fp_path, target_lookup, use_sparse=False):
    """Load fingerprint CSV and join with targets. Returns X, y.

    Parameters
    ----------
    fp_path : str
        Path to the split fingerprint CSV.
    target_lookup : dict
        molecule_chembl_id -> pchembl_value mapping.
    use_sparse : bool
        If True and bit density < 30%, return sparse matrix.
    """
    df = pd.read_csv(fp_path, dtype={'molecule_chembl_id': str})

    # Float32 for all feature columns (halves memory)
    fp_cols = [c for c in df.columns if c != 'molecule_chembl_id']
    df[fp_cols] = df[fp_cols].astype(np.float32)

    # Join with targets
    df['pchembl_value'] = df['molecule_chembl_id'].map(target_lookup)
    df = df.dropna(subset=['pchembl_value'])

    X = df[fp_cols].values
    y = df['pchembl_value'].values.astype(np.float32)

    # Use sparse if binary FP with low density
    if use_sparse:
        density = np.count_nonzero(X) / X.size
        if density < 0.30:
            X = csr_matrix(X)

    return X, y, fp_cols


# Preload target lookup (used for all fingerprints)
target_lookup = load_curated_targets(CURATED_PATH)
print(f"Loaded {len(target_lookup)} molecule targets")

## 4. Model Definitions (GPU-Aware)

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, AdaBoostRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.cross_decomposition import PLSRegression
from xgboost import XGBRegressor


def get_models(use_gpu=True):
    """Return list of (name, model, supports_sparse) tuples.

    Ordered fastest-first based on benchmarks. Slow models commented out
    to stay within ~30 min for all 24 fingerprint x split combinations.

    Approx GPU time per FP (3K rows): ~60s total for 14 active models
    24 combos x 60s = ~24 min total
    """
    gpu = use_gpu and CUML_AVAILABLE

    if gpu:
        from cuml.linear_model import Ridge as cuRidge
        from cuml.linear_model import Lasso as cuLasso
        from cuml.linear_model import ElasticNet as cuElasticNet
        from cuml.neighbors import KNeighborsRegressor as cuKNN
        from cuml.svm import SVR as cuSVR
        from cuml.ensemble import RandomForestRegressor as cuRF

    models = [
        # --- FAST (<1s per fit on GPU) ---
        ("Lasso Regression",
         cuLasso(alpha=0.1) if gpu else Lasso(alpha=0.1, random_state=RANDOM_STATE),
         not gpu),

        ("ElasticNet",
         cuElasticNet(alpha=0.1, l1_ratio=0.5) if gpu
         else ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE),
         not gpu),

        ("Decision Tree",
         DecisionTreeRegressor(random_state=RANDOM_STATE),
         True),

        ("Bayesian Ridge",
         BayesianRidge(),
         False),

        ("PLS Regression",
         PLSRegression(n_components=10),
         False),

        ("Ridge Regression",
         cuRidge(alpha=1.0) if gpu else Ridge(alpha=1.0, solver='svd'),
         not gpu),

        # --- MEDIUM (1-10s per fit on GPU) ---
        ("Gradient Boosting",
         GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
         False),

        ("XGBoost",
         XGBRegressor(n_estimators=500, random_state=RANDOM_STATE,
                      device='cuda', tree_method='hist', verbosity=0),
         True),

        ("SVR (RBF)",
         cuSVR(kernel='rbf', C=1.0, epsilon=0.1) if gpu
         else SVR(kernel='rbf', C=1.0, epsilon=0.1),
         not gpu),

        ("SVR (Linear)",
         cuSVR(kernel='linear', C=1.0, epsilon=0.1) if gpu
         else SVR(kernel='linear', C=1.0, epsilon=0.1),
         not gpu),

        ("k-Nearest Neighbors",
         cuKNN(n_neighbors=5) if gpu else KNeighborsRegressor(n_neighbors=5),
         not gpu),

        ("MLP Regressor",
         MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=1000,
                      random_state=RANDOM_STATE, early_stopping=True),
         False),

        ("AdaBoost",
         AdaBoostRegressor(n_estimators=500, random_state=RANDOM_STATE),
         False),

        # --- SLOW (10-100s per fit) ---
        ("Random Forest",
         cuRF(n_estimators=500, random_state=RANDOM_STATE) if gpu
         else RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
         not gpu),

        # --- VERY SLOW (uncomment if you have time) ---
        # Extra Trees: ~96s/fit on CPU, no cuML version available
        # ("Extra Trees",
        #  ExtraTreesRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
        #  True),

        # Gaussian Process: O(n^3) complexity, ~30-120s/fit for 3K samples
        # ("Gaussian Process",
        #  GaussianProcessRegressor(random_state=RANDOM_STATE, n_restarts_optimizer=2),
        #  False),
    ]
    return models


# Quick check
models = get_models(use_gpu=True)
gpu_models = [name for name, m, _ in models
              if hasattr(m, '__module__') and 'cuml' in getattr(m, '__module__', '')]
print(f"Total active models: {len(models)}")
print(f"GPU-accelerated (cuML): {gpu_models or 'None (sklearn fallback)'}")
print(f"XGBoost device: cuda")


## 5. Training Engine

In [ ]:
import math
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.base import clone
from scipy.sparse import issparse
from tqdm.notebook import tqdm


def compute_metrics(y_true, y_pred):
    """Compute R², RMSE, MAE."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    rmse = math.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    return r2, rmse, mae


def is_cuml_model(model):
    """Check if model is a cuML estimator."""
    module = getattr(model, '__module__', '') or ''
    return 'cuml' in module


def cuml_cross_val_predict(model, X, y, cv):
    """Manual cross_val_predict for cuML models (not sklearn-compatible CV)."""
    import cudf
    import cupy as cp

    predictions = np.zeros_like(y, dtype=np.float32)
    for train_idx, val_idx in cv.split(X):
        X_tr = cp.asarray(X[train_idx], dtype=cp.float32)
        y_tr = cp.asarray(y[train_idx], dtype=cp.float32)
        X_val = cp.asarray(X[val_idx], dtype=cp.float32)

        m = clone(model)
        m.fit(X_tr, y_tr)
        preds = m.predict(X_val)
        predictions[val_idx] = cp.asnumpy(preds).ravel()

    return predictions


def train_single_model(name, model, X_train, y_train, X_test, y_test,
                        cv, supports_sparse):
    """Train a single model, compute train/CV/test metrics. Returns dict."""
    start = time.time()

    # Convert sparse to dense for models that don't support it
    X_tr = X_train.toarray() if issparse(X_train) and not supports_sparse else X_train
    X_te = X_test.toarray() if issparse(X_test) and not supports_sparse else X_test

    # For cuML models, ensure dense float32 numpy arrays
    if is_cuml_model(model):
        X_tr = np.ascontiguousarray(X_tr if not issparse(X_tr) else X_tr.toarray(), dtype=np.float32)
        X_te = np.ascontiguousarray(X_te if not issparse(X_te) else X_te.toarray(), dtype=np.float32)
        y_tr = np.ascontiguousarray(y_train, dtype=np.float32)
    else:
        y_tr = y_train

    # Fit
    model.fit(X_tr, y_tr)

    # Predict train & test
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)

    # Flatten PLS 2D output
    if hasattr(y_pred_train, 'ndim') and y_pred_train.ndim > 1:
        y_pred_train = y_pred_train.ravel()
    if hasattr(y_pred_test, 'ndim') and y_pred_test.ndim > 1:
        y_pred_test = y_pred_test.ravel()

    # Ensure numpy arrays (cuML may return cupy)
    try:
        import cupy as cp
        if isinstance(y_pred_train, cp.ndarray):
            y_pred_train = cp.asnumpy(y_pred_train)
        if isinstance(y_pred_test, cp.ndarray):
            y_pred_test = cp.asnumpy(y_pred_test)
    except ImportError:
        pass

    # Cross-validation
    if is_cuml_model(model):
        y_pred_cv = cuml_cross_val_predict(clone(model), X_tr, y_tr, cv)
    else:
        # Use n_jobs=-1 for CPU parallelism on sklearn models
        n_jobs = -1 if not is_cuml_model(model) else 1
        model_cv = clone(model)
        y_pred_cv = cross_val_predict(model_cv, X_tr, y_tr, cv=cv, n_jobs=n_jobs)

    if hasattr(y_pred_cv, 'ndim') and y_pred_cv.ndim > 1:
        y_pred_cv = y_pred_cv.ravel()

    elapsed = time.time() - start

    # Metrics
    r2_train, rmse_train, mae_train = compute_metrics(y_train, y_pred_train)
    r2_cv, rmse_cv, mae_cv = compute_metrics(y_train, y_pred_cv)
    r2_test, rmse_test, mae_test = compute_metrics(y_test, y_pred_test)

    return {
        'train': {'model': name, 'r2': r2_train, 'rmse': rmse_train, 'mae': mae_train, 'time_s': elapsed},
        'cv': {'model': name, 'r2': r2_cv, 'rmse': rmse_cv, 'mae': mae_cv, 'time_s': elapsed},
        'test': {'model': name, 'r2': r2_test, 'rmse': rmse_test, 'mae': mae_test, 'time_s': elapsed},
    }

## 6. Main Training Loop (All Fingerprints × Splits)

In [ ]:
# Discover all fingerprint/split combinations
FINGERPRINTS = [
    'maccs', 'pubchem', 'substructure', 'klekota_roth',
    'cdk', 'cdk_ext', 'cdk_graphonly', 'estate',
    'atompairs2d', 'substructure_count', 'klekota_roth_count', 'atompairs2d_count',
]
SPLITS = ['random', 'kennard_stone']

# Binary fingerprints benefit from sparse storage
BINARY_FPS = {'maccs', 'pubchem', 'substructure', 'klekota_roth',
              'cdk', 'cdk_ext', 'cdk_graphonly', 'estate', 'atompairs2d'}


def get_split_paths(fp_name, split_name):
    """Resolve train/test file paths for a fingerprint+split combination."""
    stem = f"aromatase_{fp_name}_fp"
    train = os.path.join(SPLITS_DIR, f"{stem}_{split_name}_train.csv")
    test = os.path.join(SPLITS_DIR, f"{stem}_{split_name}_test.csv")
    return train, test


def load_completed(path):
    """Return set of model names already saved."""
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    return set(df['model'].tolist())


def append_result(path, row_dict):
    """Append a single result row to CSV (creates file if needed)."""
    fields = ['model', 'r2', 'rmse', 'mae', 'time_s']
    file_exists = os.path.exists(path) and os.path.getsize(path) > 0
    df = pd.DataFrame([row_dict], columns=fields)
    df.to_csv(path, mode='a', header=not file_exists, index=False)


# Verify all split files exist
missing = []
for fp in FINGERPRINTS:
    for split in SPLITS:
        train_p, test_p = get_split_paths(fp, split)
        if not os.path.exists(train_p):
            missing.append(f"{fp}/{split}")

if missing:
    print(f"WARNING: Missing split files for: {missing}")
else:
    print(f"All {len(FINGERPRINTS) * len(SPLITS)} fingerprint×split combinations found.")
    print(f"Total model fits: {len(FINGERPRINTS) * len(SPLITS) * 16} (+ 10-fold CV each)")

In [ ]:
# === MAIN TRAINING LOOP ===
# Iterates over all fingerprints × splits × models with checkpointing.
# Safe to re-run: skips already-completed models.

cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_models = get_models(use_gpu=True)

total_combos = len(FINGERPRINTS) * len(SPLITS)
combo_pbar = tqdm(total=total_combos, desc="FP×Split combos", position=0)

global_results = []  # Collect all results for final summary

for fp_name in FINGERPRINTS:
    for split_name in SPLITS:
        train_path, test_path = get_split_paths(fp_name, split_name)
        if not os.path.exists(train_path):
            combo_pbar.update(1)
            continue

        # Output file paths
        prefix = f"{fp_name}_{split_name}_split"
        out_train = os.path.join(OUTPUT_DIR, f"{prefix}_train.csv")
        out_cv = os.path.join(OUTPUT_DIR, f"{prefix}_cv.csv")
        out_test = os.path.join(OUTPUT_DIR, f"{prefix}_test.csv")

        # Check what's already done
        completed = load_completed(out_train)
        models_to_run = [(n, m, sp) for n, m, sp in all_models if n not in completed]

        if not models_to_run:
            combo_pbar.update(1)
            continue

        # Load data
        use_sparse = fp_name in BINARY_FPS
        X_train, y_train, fp_cols = load_split_data(train_path, target_lookup, use_sparse)
        X_test, y_test, _ = load_split_data(test_path, target_lookup, use_sparse)

        desc = f"{fp_name}/{split_name}"
        model_pbar = tqdm(models_to_run, desc=desc, position=1, leave=False)

        for name, model, supports_sparse in model_pbar:
            model_pbar.set_postfix_str(name)
            try:
                result = train_single_model(
                    name, clone(model), X_train, y_train, X_test, y_test,
                    cv, supports_sparse
                )
                # Checkpoint immediately
                append_result(out_train, result['train'])
                append_result(out_cv, result['cv'])
                append_result(out_test, result['test'])

                global_results.append({
                    'fingerprint': fp_name, 'split': split_name,
                    **result['test']
                })
            except Exception as e:
                print(f"\n  ERROR: {fp_name}/{split_name}/{name}: {e}")
                continue

        # Free memory after each fingerprint
        del X_train, y_train, X_test, y_test
        gc.collect()

        combo_pbar.update(1)

combo_pbar.close()
print(f"\nDone! All results saved to: {OUTPUT_DIR}")

## 7. Results Aggregation

In [ ]:
# Load all test results into a master DataFrame
all_test_results = []

for fp_name in FINGERPRINTS:
    for split_name in SPLITS:
        prefix = f"{fp_name}_{split_name}_split"
        test_path = os.path.join(OUTPUT_DIR, f"{prefix}_test.csv")
        if os.path.exists(test_path):
            df = pd.read_csv(test_path)
            df['fingerprint'] = fp_name
            df['split'] = split_name
            all_test_results.append(df)

if all_test_results:
    results_df = pd.concat(all_test_results, ignore_index=True)
    print(f"Total results: {len(results_df)} (models × FP × splits)")
    print(f"\nTop 20 models by test R²:")
    top20 = results_df.nlargest(20, 'r2')[['fingerprint', 'split', 'model', 'r2', 'rmse', 'mae', 'time_s']]
    display(top20.reset_index(drop=True))
else:
    print("No results found yet. Run the training loop first.")

In [ ]:
# Best model per fingerprint
if all_test_results:
    best_per_fp = results_df.loc[
        results_df.groupby(['fingerprint', 'split'])['r2'].idxmax()
    ][['fingerprint', 'split', 'model', 'r2', 'rmse']].sort_values('r2', ascending=False)
    print("Best model per fingerprint × split:")
    display(best_per_fp.reset_index(drop=True))

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_test_results:
    # --- Heatmap: R²(test) for random split ---
    random_df = results_df[results_df['split'] == 'random']
    pivot = random_df.pivot_table(index='model', columns='fingerprint', values='r2')

    # Order by mean R² across fingerprints
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0.4,
                vmin=0, vmax=0.8, ax=ax, linewidths=0.5)
    ax.set_title('Test R² — Random Split (all fingerprints × models)', fontsize=13)
    ax.set_xlabel('Fingerprint')
    ax.set_ylabel('Model')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'heatmap_r2_random.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # --- Heatmap: R²(test) for Kennard-Stone split ---
    ks_df = results_df[results_df['split'] == 'kennard_stone']
    if not ks_df.empty:
        pivot_ks = ks_df.pivot_table(index='model', columns='fingerprint', values='r2')
        pivot_ks = pivot_ks.loc[pivot_ks.mean(axis=1).sort_values(ascending=False).index]

        fig, ax = plt.subplots(figsize=(14, 8))
        sns.heatmap(pivot_ks, annot=True, fmt='.3f', cmap='RdYlGn', center=0.4,
                    vmin=0, vmax=0.8, ax=ax, linewidths=0.5)
        ax.set_title('Test R² — Kennard-Stone Split (all fingerprints × models)', fontsize=13)
        ax.set_xlabel('Fingerprint')
        ax.set_ylabel('Model')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'heatmap_r2_kennard_stone.png'), dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
if all_test_results:
    # --- Runtime comparison ---
    time_df = results_df[results_df['split'] == 'random'].groupby('model')['time_s'].mean().sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))
    time_df.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Average time per model (seconds)')
    ax.set_title('Model Training Time (avg across fingerprints, random split)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'runtime_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # --- Random vs Kennard-Stone comparison ---
    comparison = results_df.groupby(['model', 'split'])['r2'].mean().unstack()
    if 'kennard_stone' in comparison.columns:
        comparison['diff'] = comparison['random'] - comparison['kennard_stone']
        comparison = comparison.sort_values('random', ascending=False)
        print("\nRandom vs Kennard-Stone (mean R² across fingerprints):")
        display(comparison.round(4))

In [ ]:
if all_test_results:
    # Save master results table
    master_path = os.path.join(OUTPUT_DIR, 'all_results_master.csv')
    results_df.to_csv(master_path, index=False)
    print(f"Master results saved to: {master_path}")
    print(f"Shape: {results_df.shape}")

In [ ]:
# === Download Results as ZIP ===
import zipfile
from google.colab import files

zip_name = 'regression_model_results.zip'

print(f"Zipping results from: {OUTPUT_DIR}")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(OUTPUT_DIR):
        for fname in sorted(filenames):
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, REPO_DIR)
            zf.write(fpath, arcname)

    print(f"\nContents ({len(zf.infolist())} files):")
    for info in zf.infolist():
        print(f"  {info.filename} ({info.file_size/1024:.1f} KB)")

print(f"\n{zip_name} ({os.path.getsize(zip_name)/1024:.1f} KB)")
files.download(zip_name)